In [1]:
import pandas as pd
import numpy as np 

# csv파일을 불러옴
df1 = pd.read_csv('../data/tmdb_5000_credits.csv')
df2 = pd.read_csv('../data/tmdb_5000_movies.csv')
df1.columns = ['id', 'title', 'cast', 'crew']

# 두 csv파일을 하나로 합침(중복된 제목은 빼고)
df = df2.merge(df1[['id', 'cast', 'crew']], on='id')

In [2]:
from ast import literal_eval
# 장르의 값은 타입이 문자열
type(df['genres'][0])
# 장르의 값을 리스트로 변경
genre0 = literal_eval(df['genres'][0])
# genre0
# for genre in genre0:
# 	print(genre['name'])

In [3]:
# cast와 같은 타입이 문자열
type(df['cast'][0])
# cast의 값을 리스트로 변경
cast0 = literal_eval(df['cast'][0])
# cast0 # cast_id, character, gender, id, name, order
# for cast in cast0:
# 	print(cast['name'])

In [4]:
# keyword와 같은 타입이 문자열
type(df['keywords'][0])
# keword의 값을 리스트로 변경
keyword0 = literal_eval(df['keywords'][0])
# keyword0
# keyword0 # id, name
lst = []
for keyword in keyword0:
	# print(keyword['name'])
	lst.append(keyword['name'])

In [5]:
# x데이터가 들어오면 x데이터에서 name들만 추출하여 리스트를 만들어 반환
# 예 : cast 데이터가 들어오면 cast 데이터에서 배우 이름만 리스트로 만들어 반환
def get_list(x):
	# x가 리스트인지 확인 
	if isinstance(x, list):
		# names = []
		# for i in x:
		# 	names.append(i['name'])
		names = [i['name'] for i in x]
		# 처음부터 3개만 추출(최대 3개만 추출)
		if len(names) > 3:
			names = names[:3] # 0번지부터 3번지 전까지
		return names
	return []

In [6]:
# genres, keywords, cast
get_list(literal_eval(df['genres'][0]))

['Action', 'Adventure', 'Fantasy']

In [7]:
features = ['genres', 'keywords', 'cast', 'crew']
# 문자열로 된 값을 리스트로 변환
for feature in features:
	df[feature] = df[feature].apply(literal_eval)

In [8]:
# df['genres'][0]

In [9]:
# genres, keywords, cast를 name만 추출해서 덮어쓰기
# df['genres'] = df['genres'].apply(literal_eval)
# df['genres'] = df['genres'].apply(get_list)
features = ['genres', 'keywords', 'cast']
for feature in features:
	df[feature] = df[feature].apply(get_list)


In [10]:
# df['genres']

In [11]:
# cast, genres, keywords 리스트 값에 있는 공백을 제거 
# 공백을 제거하지 않고 나~중에 학습을 하면
# 배우 성때문에 해당 배우와 상관없는 영화가 추천되거나
# 공상 과학에서 과학 때문에 순수 과학 영화가 추천되는 상황이 발생할 수 있음

In [12]:
sample = df['cast'][0]
print(sample)
result = [str.lower(i) for i in sample]
print(result)
result = [i.replace(' ', '') for i in sample]
print(result)

['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']
['sam worthington', 'zoe saldana', 'sigourney weaver']
['SamWorthington', 'ZoeSaldana', 'SigourneyWeaver']


In [13]:
# 리스트 또는 문자열에 있는 공백을 제거하고 소문자로 만들어 주는 함수
def clean_data(x):
	if isinstance(x, list):
		return [str.lower(i.replace(' ', '')) for i in x]
	elif isinstance(x, str):
		return [str.lower(x.replace(' ', ''))]
	else:
		return ''

In [14]:
for feature in features:
	df[feature] = df[feature].apply(clean_data)

In [15]:
df['cast'][0]

['samworthington', 'zoesaldana', 'sigourneyweaver']

In [16]:
jobs = [i['job'] for i in df['crew'][0]]
# jobs # Director : 감독

In [17]:
# 0번 영화 Avatar의 스탭들 중 감독을 조회
for crew in df['crew'][0]:
	if crew['job'] == 'Director':
		print(crew['name'])

James Cameron


In [18]:
# crew 정보를 주면 감독을 추출하는 함수
def get_director(x):
	for crew in x:
		if crew['job'] == 'Director':
			return crew['name']
	return np.nan

In [19]:
get_director(df['crew'][0])

'James Cameron'

In [20]:
df['director'] = df['crew'].apply(get_director)
df['director'] = df['director'].apply(clean_data)

In [21]:
# 감독, 배우, 키워드, 장르를 하나의 열에 통합
x = df.loc[0]
# f'{} {} {} {}
res = f'{' '.join(x['keywords'])} {' '.join(x['keywords'])} {' '.join(x['genres'])} {x['director']}'
res
# print(' '.join(x['keywords']))
# print(' '.join(x['cast']))
# print(' '.join(x['genres']))
# print(x['director'])

"cultureclash future spacewar cultureclash future spacewar action adventure fantasy ['jamescameron']"

In [22]:
# 감독, 배우, 키워드, 장르를 하나의 열에 통합하는 함수
def create_soup(x):
	return f'{' '.join(x['keywords'])} {' '.join(x['keywords'])} {' '.join(x['genres'])} {x['director']}'

In [23]:
df['soup'] = df.apply(create_soup, axis=1)
# df['soup']

In [24]:
# 글자 수를 계산
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer()
count_matrix = count.fit_transform(df['soup'])

In [25]:
# 코사인 유사도 계산
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(count_matrix, count_matrix)

In [26]:
indices = pd.Series(df.index, index=df2['title']).drop_duplicates()
def get_recommendations(title, cosine_sim=cosine_sim):
	# 영화 제목으로 idx를 찾음(왜? 해당 영화의 코사인 유사도를 가져오기 위해)
	# 코사인 유사도는 idx를 이용해서 찾아야 함
	idx = indices[title]
	# 코사인 유사도를 정렬하려고 하는데 그냥 정렬하면 기존 위치가 섞임 
	# => 유사도만 보면 어떤 영화인지 모름
	# => 코사인 유도를 (idx, 코사인유사도값)으로 된 튜플을 만들면 섞여도 idx를 알 수 있음
	sim_scores = list(enumerate(cosine_sim[idx]))
	# 코사인 유사도를 기준으로 정렬(람다함수 활용)
	sim_scores = sorted(sim_scores, key=lambda x : x[1], reverse=True)
	# 0번지는 검색 영화이기 때문에 0번지를 제외한 10개 추출
	sim_scores = sim_scores[1:11]
	# 여기서부터는 코사인 유사도 값이 필요없기 때문에 영화 idx만 추출해서 리스트로 만듬
	movie_indices = [i[0] for i in sim_scores]
	# 영화 idx를 이용해서 영화 제목들을 가져와서 반환
	return df['title'].iloc[movie_indices]

In [27]:
print(get_recommendations('Avatar', cosine_sim))

466                            The Time Machine
71        The Mummy: Tomb of the Dragon Emperor
85          Captain America: The Winter Soldier
127                          Mad Max: Fury Road
2444                            Damnation Alley
224                                     RoboCop
3854    Batman: The Dark Knight Returns, Part 2
4114                                     Subway
487                                  Red Planet
812                                  Pocahontas
Name: title, dtype: object
